In [1]:
import pandas as pd
import numpy as np
import os
from model_diffusion import eval_bagging

In [2]:
all_df = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/disgent_2020/timecut/dga_time_uniport.csv')
merged_df = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/input_deep_svd/node2vec_features.csv')
all_df = all_df[all_df['string_id'].isin(merged_df['string_id'])]

In [5]:
time = 2019
selected_diseases = []
for disease_id in all_df['disease_id'].unique():
    sub_df = all_df[all_df['disease_id']==disease_id]
    if len(sub_df) < 15:
        continue
    else:
        # print(type(time),type(sub_df['first_pub_year'].max()))
        if sub_df['first_pub_year'].max() > time and sub_df['first_pub_year'].min() <= time and len(sub_df[sub_df['first_pub_year']<time]) >=5:
            selected_diseases.append(disease_id)

In [17]:
summary = all_df[all_df['disease_id'].isin(selected_diseases)][['disease_id','disease_name']].drop_duplicates()


In [19]:
num = []
for disease in summary['disease_id'].unique():
    num.append(len(all_df[all_df['disease_id']==disease]))

In [20]:
summary['Gene_number'] = num

In [21]:
summary

,disease_id,disease_name,Gene_number
93,ICD10_C16,Malignant neoplasm of stomach,90
183,ICD10_C18,Malignant neoplasm of colon,50
216,ICD10_C43,Malignant melanoma of skin,47
439,ICD10_C50,"Cancer, Breast",544
1607,ICD10_C67,Malignant neoplasm of bladder,191
1894,ICD10_C81,Hodgkin's granuloma,23
2087,ICD10_D57,"Anemia, Sickle Cell",22
2115,ICD10_D66,Hemophilia A,15
2149,ICD10_D83,Common Variable Immunodeficiency,23
2286,ICD10_E11,Adult-Onset Diabetes Mellitus,269


In [23]:
summary.to_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/input_deep_svd/main_data/train_data/2019/disease_summary.csv',index=False)

In [26]:
root = '/itf-fi-ml/shared/users/ziyuzh/svm/data/input_deep_svd/main_data/train_data/2019'

for disease in selected_diseases:
    eg_df = all_df[all_df['disease_id']==disease]
    train = eg_df[eg_df['first_pub_year']<time]
    train = train[['disease_id', 'disease_name', 'gene_id', 'string_id']]
    train = train.rename(columns={'string_id': 'ensembl','gene_id': 'gene_symbol'})
    test = eg_df[eg_df['first_pub_year']>=time]
    test = test[['disease_id', 'disease_name', 'gene_id', 'string_id']]
    test = test.rename(columns={'string_id': 'ensembl','gene_id': 'gene_symbol'})
    train.to_csv(os.path.join(root,'train',disease+'.csv'),index=False)
    test.to_csv(os.path.join(root,'test',disease+'.csv'),index=False)

In [74]:
root = '/itf-fi-ml/shared/users/ziyuzh/svm/data/input_deep_svd/train'
for disease in all_df['disease_id'].unique():
    all_df[all_df['disease_id']==disease].to_csv(os.path.join(root,disease+'.csv'),index=False)

# no remap

In [2]:
import pickle
import pandas as pd

root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/occ_deep_svd/scores'
out_path = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2019_occ_deep_svd'
out_path_pred = out_path+'_pred/pred.pkl'

full_path = '/itf-fi-ml/shared/users/ziyuzh/svm/results/occ_deep_svd/scores/ICD10_M41_scores_2019z.csv'

score_df = pd.read_csv(full_path)
score_df['score'] = 1-score_df['score']
disease = 'ICD10_M41'

result_df = pd.DataFrame(columns=['method',"fold","para", 'top_recall_25','top_recall_300','top_recall_10%', 'top_precision_10%', 'max_precision_10%','top_recall_30%', 'top_precision_30%', 'max_precision_30%','pm_0.5%','pm_1%','pm_5%','pm_10%','pm_15%','pm_20%','pm_25%','pm_30%','auroc',"rank_ratio",'bedroc_1','bedroc_5','bedroc_10','bedroc_30'])
predcition_collection = dict()

ranked_predict_index, results = eval_bagging(score_df['score'].to_numpy(), score_df['y_true'].to_numpy())

result_df.loc[len(result_df.index)] = ["occ_deep_svd",1,'uniport_ppi_2019'+'-0-0-0', *results]
# predcition_collection['true_label'] = score_df['y_true'].to_numpy()
# predcition_collection["test_genes"] = score_df['gene_id'].to_list()
# predcition_collection['uniport_ppi_2019'] = score_df['score'].to_numpy()

# with open(out_path_pred+f'/{disease}_pred.pkl', 'wb') as f:
#     pickle.dump(predcition_collection, f)

result_df.to_csv(os.path.join(out_path, f"{disease}.csv"),index = False)

In [13]:
predcition_collection['test_genes'] =pd.Index(predcition_collection['test_genes'])

In [14]:
type(predcition_collection['test_genes'])

pandas.core.indexes.base.Index

In [15]:
predcition_collection['test_genes'][predcition_collection['true_label'] == 1]

Index(['Q92581', 'Q12805'], dtype='object')

In [ ]:
predcition_collection['true_label'].sum(),predcition_collection['test_genes']

with open(out_path_pred+f'/{disease}_pred.pkl', 'wb') as f:
    pickle.dump(predcition_collection, f)

(2,
 Index(['P06400', 'P27361', 'Q06418', 'P05412', 'Q09472', 'Q04864', 'O60674',
        'P05019', 'P12931', 'P28482',
        ...
        'Q7Z7B7', 'O14523', 'Q8NDH6', 'Q8NAX2', 'Q8TE49', 'O76001', 'Q14533',
        'Q8WWB7', 'Q5TGU0', 'O95484'],
       dtype='object', length=15661))

In [ ]:
import pickle

root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/occ_deep_svd/scores'
out_path = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2019_occ_deep_svd'
out_path_pred = out_path+'_pred/pred.pkl'
os.makedirs(out_path, exist_ok=True)
os.makedirs(out_path_pred, exist_ok=True)

for file in os.listdir(root):
    full_path = os.path.join(root, file)
    
    # Skip directories
    if os.path.isdir(full_path):
        continue
    
    # Ensure it's a CSV file
    if not file.endswith(".csv"):
        continue
    if not file.startswith("ICD"):
        continue

    score_df = pd.read_csv(full_path)
    score_df['score'] = 1-score_df['score']
    disease = file.split('.')[0][:9]
    # rest of your code...

    result_df = pd.DataFrame(columns=['method',"fold","para", 'top_recall_25','top_recall_300','top_recall_10%', 'top_precision_10%', 'max_precision_10%','top_recall_30%', 'top_precision_30%', 'max_precision_30%','pm_0.5%','pm_1%','pm_5%','pm_10%','pm_15%','pm_20%','pm_25%','pm_30%','auroc',"rank_ratio",'bedroc_1','bedroc_5','bedroc_10','bedroc_30'])
    predcition_collection = dict()
    
    ranked_predict_index, results = eval_bagging(score_df['score'].to_numpy(), score_df['y_true'].to_numpy())

    result_df.loc[len(result_df.index)] = ["occ_deep_svd",1,'uniport_ppi_2019'+'-0-0-0', *results]
    predcition_collection['true_label'] = score_df['y_true'].to_numpy()
    predcition_collection["test_genes"] = score_df['gene_id'].to_list()
    predcition_collection['uniport_ppi_2019'] = score_df['score'].to_numpy()

    with open(out_path_pred+f'/{disease}_pred.pkl', 'wb') as f:
        pickle.dump(predcition_collection, f)

    result_df.to_csv(os.path.join(out_path, f"{disease}.csv"),index = False)

In [26]:
result_df

,method,fold,para,top_recall_25,top_recall_300,top_recall_10%,top_precision_10%,max_precision_10%,top_recall_30%,top_precision_30%,...,pm_15%,pm_20%,pm_25%,pm_30%,auroc,rank_ratio,bedroc_1,bedroc_5,bedroc_10,bedroc_30
0,occ_deep_svd,1,uniport_ppi_2019-0-0-0,0.0,0.25,0.75,0.001914,0.002553,0.75,0.000638,...,0.83349,0.78959,0.750132,0.714386,0.90642,0.0937,0.19591,0.402087,0.529772,0.711336


In [18]:
score_df = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/results/occ_deep_svd/scores/ICD10_C16_scores_2019z.csv')

In [24]:
score_df[score_df['y_true']==1]['score'].mean()

0.0007743102704317577

In [23]:
score_df['score'].mean()

0.0016839118473106031

# with remap(doesn't work)

In [38]:
root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/occ_deep_svd/scores'
all_genes = set()
all_dict = dict()
for file in os.listdir(root):
    if '2019' in file:
        resulst = pd.read_csv(os.path.join(root,file))
        all_genes.update(resulst['gene_id'].tolist())
        disease = file.split('.')[0][5:14]
        all_dict[disease] = resulst
all_genes = [item.split('.')[-1] for item in all_genes]

In [21]:
import mygene
def get_map_df(ensembl_ids,input_type):
    mg = mygene.MyGeneInfo()
    # Query mygene for UniProt and Entrez gene ID mappings
    results = mg.querymany(
        ensembl_ids,
        scopes=input_type,
        fields='uniprot',
        species='human'
    )

    results_df = pd.DataFrame(results)
    results_df['uniprot_ids'] = results_df['uniprot'].apply(
        lambda x: list(x.values())[0] if isinstance(x, dict) and 'Swiss-Prot' in x else None)
    results_df = results_df[~results_df['uniprot_ids'].isna()]
    results_df[results_df['uniprot_ids'].apply(lambda x: isinstance(x, list) and len(x) > 1)]
    return results_df

ppi_ids_map = get_map_df(all_genes,'ensembl.protein')

ppi_set = set()
for values in ppi_ids_map['uniprot_ids']:
    if isinstance(values, list) and len(values) > 1:
        ppi_set.update(values)  # Add all elements in the list
    else:
        ppi_set.add(values)

string_ids = []
one2more = []
more2one = []  # to collect subdfs with multiple or zero matches

for uniport_ids in list(ppi_set):
    subdf = ppi_ids_map[ppi_ids_map['uniprot_ids'].str.contains(uniport_ids, na=False)]
    
    if len(subdf) == 1:
        if isinstance(subdf['uniprot_ids'], list) and len(values) > 1:
            one2more.append(subdf)
        else:
            string_ids.append(uniport_ids)
    else:
        more2one.append(subdf)

more2one_df = pd.concat(more2one, ignore_index=True)

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
2 input query terms found dup hits:	[('ENSP00000319979', 2), ('ENSP00000473163', 3)]
1228 input query terms found no hit:	['ENSP00000398177', 'ENSP00000237449', 'ENSP00000349525', 'ENSP00000375748', 'ENSP00000441802', 'ENS


In [37]:
all_dict.keys()

dict_keys(['ICD10_I46', 'ICD10_L80', 'ICD10_K51', 'ICD10_K44', 'ICD10_G43', 'ICD10_D83', 'ICD10_G40', 'ICD10_G24', 'ICD10_M41', 'ICD10_G30', 'ICD10_E84', 'ICD10_C56', 'ICD10_E11', 'ICD10_C67', 'ICD10_M32', 'ICD10_C50', 'ICD10_G10', 'ICD10_F20', 'ICD10_C18', 'ICD10_I10', 'ICD10_G91', 'ICD10_C16', 'ICD10_F31', 'ICD10_L40', 'ICD10_I42', 'ICD10_I63', 'ICD10_I70', 'ICD10_N17', 'ICD10_I50', 'ICD10_D57', 'ICD10_E66', 'ICD10_C43', 'ICD10_F90', 'ICD10_N46', 'ICD10_G20', 'ICD10_N97', '_M41_scor', 'ICD10_N18', 'ICD10_F72', 'ICD10_N04', 'ICD10_J80', 'ICD10_I25', 'ICD10_C81'])

In [41]:
all_dict['ICD10_I46']

,gene_id,y_true,score
0,9606.ENSP00000252729,0,0.000748
1,9606.ENSP00000377840,0,0.000941
2,9606.ENSP00000372169,0,0.001014
3,9606.ENSP00000226021,0,0.001117
4,9606.ENSP00000266376,0,0.001366
...,...,...,...
15135,9606.ENSP00000381272,0,0.091443
15136,9606.ENSP00000318395,0,0.095798
15137,9606.ENSP00000230012,0,0.096502
15138,9606.ENSP00000276127,0,0.097154


In [60]:
import pickle


In [61]:

for disease in all_dict.keys():

    out_path = os.path.join(root,'results/2019_occ_deep_svd')
    out_path_pred = out_path+'_pred/pred.pkl'
    os.makedirs(out_path, exist_ok=True)
    os.makedirs(out_path_pred, exist_ok=True)

    result_df = pd.DataFrame(columns=['method',"fold","para", 'top_recall_25','top_recall_300','top_recall_10%', 'top_precision_10%', 'max_precision_10%','top_recall_30%', 'top_precision_30%', 'max_precision_30%','pm_0.5%','pm_1%','pm_5%','pm_10%','pm_15%','pm_20%','pm_25%','pm_30%','auroc',"rank_ratio",'bedroc_1','bedroc_5','bedroc_10','bedroc_30'])
    predcition_collection = dict()
    ppi_emb = all_dict[disease].copy()
    ppi_emb = ppi_emb.rename(columns={'gene_id': 'string_id'})
    aggregated_rows = []

    # Iterate over each UniProt ID group
    for protein_id, subdf in more2one_df.groupby('uniprot_ids'):
        # Get list of ENSP IDs
        ensp_ids = subdf['query'].tolist()

        # Add '9606.' prefix to each ENSP ID
        ensp_ids = ['9606.' + ensp_id for ensp_id in ensp_ids]

        # Select corresponding rows from ppi_emb where 'string_id' is in ensp_ids
        matched_ppi = ppi_emb[ppi_emb['string_id'].isin(ensp_ids)]

        if not matched_ppi.empty:
            # Calculate the mean of all feature columns (exclude 'string_id')
            mean_features = matched_ppi.drop(columns=['string_id']).mean()

            # Create a new row with UniProt ID and the averaged features
            mean_features['string_id'] = protein_id

            # Add to the results list
            aggregated_rows.append(mean_features)

    # Convert the list of Series into a DataFrame
    aggregated_df = pd.DataFrame(aggregated_rows)

    # Optional: Reorder columns to have 'uniprot_id' first
    cols = ['string_id'] + [col for col in aggregated_df.columns if col != 'string_id']
    aggregated_df = aggregated_df[cols]

    # Step 1: Prepare ENSP IDs with '9606.' prefix
    ensp_ids = ['9606.' + ensp_id for ensp_id in ppi_ids_map[ppi_ids_map['uniprot_ids'].isin(string_ids)]['query'].tolist()]

    # Step 2: Select matching rows from ppi_emb
    other_ppi = ppi_emb[ppi_emb['string_id'].isin(ensp_ids)].copy()

    # Step 3: Map 'string_id' back to 'uniprot_ids'
    # First, create mapping from ENSP ID with '9606.' prefix to UniProt ID
    ensp_to_uniprot = ppi_ids_map[ppi_ids_map['uniprot_ids'].isin(string_ids)].set_index('query')['uniprot_ids'].to_dict()

    # Apply mapping to refill 'string_id' with corresponding UniProt ID
    other_ppi['string_id'] = other_ppi['string_id'].apply(lambda x: ensp_to_uniprot[x.replace('9606.', '')])

    ppi_df = pd.concat([other_ppi, aggregated_df], ignore_index=True)
    if len(ppi_df['y_true'].unique()) > 1:
        ppi_df['y_true'] = (ppi_df['y_true'] > 0).astype(int)

        ranked_predict_index, results = eval_bagging(ppi_df['score'].to_numpy(), ppi_df['y_true'].to_numpy())

        result_df.loc[len(result_df.index)] = ["occ_deep_svd",1,'uniport_ppi_2019'+'-0-0-0', *results]
        predcition_collection['true_label'] = ppi_df['y_true'].to_numpy()
        predcition_collection["test_genes"] = ppi_df['string_id'].to_list()
        predcition_collection['uniport_ppi_2019'] = ppi_df['score'].to_numpy()

        with open(out_path_pred+f'/{disease}_pred.pkl', 'wb') as f:
            pickle.dump(predcition_collection, f)

        result_df.to_csv(os.path.join(out_path, f"{disease}.csv"),index = False)
        break



In [62]:
result_df

,method,fold,para,top_recall_25,top_recall_300,top_recall_10%,top_precision_10%,max_precision_10%,top_recall_30%,top_precision_30%,...,pm_15%,pm_20%,pm_25%,pm_30%,auroc,rank_ratio,bedroc_1,bedroc_5,bedroc_10,bedroc_30
0,occ_deep_svd,1,uniport_ppi_2019-0-0-0,0.0,0.0,0.0,0.0,0.004409,0.0,0.0,...,0.0,0.0,0.0,0.0,0.104667,0.8952,3.836037e-58,1.423613e-12,7.874783e-07,0.00422


In [55]:
ppi_emb

In [63]:
ppi_ids_map

,query,_id,_score,uniprot,notfound,uniprot_ids
0,ENSP00000299206,27343,29.85333,"{'Swiss-Prot': 'Q9UGP5', 'TrEMBL': ['A8K860', ...",NaN,Q9UGP5
1,ENSP00000351602,2526,29.85333,{'Swiss-Prot': 'P22083'},NaN,P22083
2,ENSP00000350009,23032,29.85333,"{'Swiss-Prot': 'Q8TEY7', 'TrEMBL': ['E9PQP0', ...",NaN,Q8TEY7
3,ENSP00000343318,10317,29.85333,"{'Swiss-Prot': 'Q9Y2C3', 'TrEMBL': 'A0A0A0MS93'}",NaN,Q9Y2C3
4,ENSP00000362527,149076,29.85333,"{'Swiss-Prot': 'Q5T0B9', 'TrEMBL': 'F5H055'}",NaN,Q5T0B9
...,...,...,...,...,...,...
15151,ENSP00000262887,7515,29.85333,"{'Swiss-Prot': 'P18887', 'TrEMBL': ['Q59HH7', ...",NaN,P18887
15152,ENSP00000302846,5734,29.85333,"{'Swiss-Prot': 'P35408', 'TrEMBL': 'A0PJF5'}",NaN,P35408
15153,ENSP00000218089,10735,29.85333,"{'Swiss-Prot': 'Q8N3U4', 'TrEMBL': ['E7ERE6', ...",NaN,Q8N3U4
15154,ENSP00000307900,2752,29.85333,"{'Swiss-Prot': 'P15104', 'TrEMBL': ['A0A2R8YDT...",NaN,P15104


In [53]:
ppi_emb = all_dict[key]
# ppi_emb = ppi_emb.rename(columns={'gene_id': 'string_id'},inplace=True)
ppi_emb

,string_id,y_true,score
0,9606.ENSP00000252729,0,0.000748
1,9606.ENSP00000377840,0,0.000941
2,9606.ENSP00000372169,0,0.001014
3,9606.ENSP00000226021,0,0.001117
4,9606.ENSP00000266376,0,0.001366
...,...,...,...
15135,9606.ENSP00000381272,0,0.091443
15136,9606.ENSP00000318395,0,0.095798
15137,9606.ENSP00000230012,0,0.096502
15138,9606.ENSP00000276127,0,0.097154
